# Voronoi and Proportional Cartogram Performance

Runtime of the Voronoi cartogram and of the proportional cartogram partitioning (`shrink` and `split`) on the US states and the congressional districts. The Voronoi cartogram uses the default backend and weights by population. Each case reports the mean over the measured rounds after one warm-up round. Results are hardware-dependent and only comparable within one run.


In [ ]:
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

_here = os.getcwd()
_json_path = next(
    (
        p
        for p in (
            os.path.join(_here, "benchmark_results.json"),
            os.path.join(_here, "docs", "performance", "benchmark_results.json"),
        )
        if os.path.exists(p)
    ),
    None,
)

_rows = []
if _json_path is not None and os.path.getsize(_json_path) > 0:
    with open(_json_path) as f:
        _data = json.load(f)
    for b in _data["benchmarks"]:
        extra = b.get("extra_info") or {}
        _rows.append({
            "name": b["name"],
            "config": b["name"].partition("[")[2].rstrip("]"),
            "n_regions": extra.get("n_regions"),
            "mean_s": b["stats"]["mean"],
            "stddev_s": b["stats"]["stddev"],
            "rounds": b["stats"]["rounds"],
        })
df_all = pd.DataFrame(_rows, columns=["name", "config", "n_regions", "mean_s", "stddev_s", "rounds"])
_DATASETS = {"us_states_gdf": "US States", "us_congressional_districts_gdf": "Congressional Districts"}


def _runtime_table(prefix, label_col):
    """Rows of one benchmark test, with the dataset and variant taken from its id."""
    df = df_all[df_all["name"].str.startswith(prefix)].copy()
    if df.empty:
        return df
    df["dataset"] = df["config"].map(lambda c: _DATASETS.get(c.split("-")[0], c.split("-")[0]))
    df[label_col] = df["config"].map(lambda c: c.split("-", 1)[1] if "-" in c else "default")
    return df


def _runtime_plot(df, label_col):
    _, ax = plt.subplots(figsize=(5, 4))
    sns.barplot(data=df, x="dataset", y="mean_s", hue=label_col, ax=ax)
    ax.set_xlabel("")
    ax.set_ylabel("mean runtime (s)")
    ax.set_yscale("log")
    plt.show()

## Voronoi cartogram

`create_voronoi_cartogram` with the default backend, weighted by population.


In [ ]:
df_v = _runtime_table("test_voronoi_runtime", "variant")
if df_v.empty:
    print("No Voronoi benchmark results in benchmark_results.json yet.")
else:
    display(df_v[["dataset", "n_regions", "mean_s", "stddev_s", "rounds"]].round(3))
    _runtime_plot(df_v, "variant")

## Proportional cartogram

`partition_geometries` with `method="shrink"` and `method="split"`, one fraction column.


In [ ]:
df_p = _runtime_table("test_partition_runtime", "method")
if df_p.empty:
    print("No proportional cartogram benchmark results in benchmark_results.json yet.")
else:
    display(df_p[["dataset", "method", "n_regions", "mean_s", "stddev_s", "rounds"]].round(3))
    _runtime_plot(df_p, "method")